In [1]:
import pandas as pd
import numpy as np

# Load the data files
boning_df = pd.read_csv('ampc2/boning.csv')  # Adjust filename as needed
slicing_df = pd.read_csv('ampc2/slicing.csv')  # Adjust filename as needed

# Your student ID ends with 5, so you need:
# Column set 1: Right Upper Leg (x,y,z)
# Column set 2: Left Upper Leg (x,y,z)

# Extract the relevant columns from both datasets
columns_to_extract = ['Frame', 
                     'Right Upper Leg x', 'Right Upper Leg y', 'Right Upper Leg z',
                     'Left Upper Leg x', 'Left Upper Leg y', 'Left Upper Leg z']

boning_extracted = boning_df[columns_to_extract].copy()
slicing_extracted = slicing_df[columns_to_extract].copy()

# Add class labels
boning_extracted['class'] = 0  # boning
slicing_extracted['class'] = 1  # slicing

# Combine the datasets
combined_data = pd.concat([boning_extracted, slicing_extracted], ignore_index=True)

# Save the combined data
combined_data.to_csv('step1_data.csv', index=False)

print(f"Combined data shape: {combined_data.shape}")
print(combined_data.head())

Combined data shape: (72060, 8)
   Frame  Right Upper Leg x  Right Upper Leg y  Right Upper Leg z  \
0      0           0.163942          -0.001378           0.045835   
1      1          -0.190849           0.214363           0.051633   
2      2          -0.054696          -0.127660          -0.076814   
3      3           0.184245          -0.046103           0.073129   
4      4           0.302372          -0.010540           0.076275   

   Left Upper Leg x  Left Upper Leg y  Left Upper Leg z  class  
0          0.008039         -0.054908          0.011806      0  
1         -0.541907          0.175742          0.231347      0  
2          0.110754          0.134195          0.062618      0  
3          0.013505         -0.010772          0.006895      0  
4          0.210097          0.026873          0.017431      0  


In [2]:
# Load data from step 1
data = pd.read_csv('step1_data.csv')

# Renaming columns to make them easier to work with
data = data.rename(columns={
    'Frame': 'frame',
    'Right Upper Leg x': 'right_leg_x',
    'Right Upper Leg y': 'right_leg_y',
    'Right Upper Leg z': 'right_leg_z',
    'Left Upper Leg x': 'left_leg_x',
    'Left Upper Leg y': 'left_leg_y',
    'Left Upper Leg z': 'left_leg_z'
})

# 1. Root mean square value of x and y
data['rms_xy_right'] = np.sqrt((data['right_leg_x']**2 + data['right_leg_y']**2) / 2)
data['rms_xy_left'] = np.sqrt((data['left_leg_x']**2 + data['left_leg_y']**2) / 2)

# 2. Root mean square value of y and z
data['rms_yz_right'] = np.sqrt((data['right_leg_y']**2 + data['right_leg_z']**2) / 2)
data['rms_yz_left'] = np.sqrt((data['left_leg_y']**2 + data['left_leg_z']**2) / 2)

# 3. Root mean square value of z and x
data['rms_zx_right'] = np.sqrt((data['right_leg_z']**2 + data['right_leg_x']**2) / 2)
data['rms_zx_left'] = np.sqrt((data['left_leg_z']**2 + data['left_leg_x']**2) / 2)

# 4. Root mean square value of x, y and z
data['rms_xyz_right'] = np.sqrt((data['right_leg_x']**2 + data['right_leg_y']**2 + data['right_leg_z']**2) / 3)
data['rms_xyz_left'] = np.sqrt((data['left_leg_x']**2 + data['left_leg_y']**2 + data['left_leg_z']**2) / 3)

# 5. Roll calculation
data['roll_right'] = 180 * np.arctan2(data['right_leg_y'], 
                                     np.sqrt(data['right_leg_x']**2 + data['right_leg_z']**2)) / np.pi
data['roll_left'] = 180 * np.arctan2(data['left_leg_y'], 
                                    np.sqrt(data['left_leg_x']**2 + data['left_leg_z']**2)) / np.pi

# 6. Pitch calculation
data['pitch_right'] = 180 * np.arctan2(data['right_leg_x'], 
                                      np.sqrt(data['right_leg_y']**2 + data['right_leg_z']**2)) / np.pi
data['pitch_left'] = 180 * np.arctan2(data['left_leg_x'], 
                                     np.sqrt(data['left_leg_y']**2 + data['left_leg_z']**2)) / np.pi

# Save the enhanced dataset
data.to_csv('step2_data.csv', index=False)

print(f"Step 2 data shape: {data.shape}")
print("Columns:", data.columns.tolist())
print(data.head())

Step 2 data shape: (72060, 20)
Columns: ['frame', 'right_leg_x', 'right_leg_y', 'right_leg_z', 'left_leg_x', 'left_leg_y', 'left_leg_z', 'class', 'rms_xy_right', 'rms_xy_left', 'rms_yz_right', 'rms_yz_left', 'rms_zx_right', 'rms_zx_left', 'rms_xyz_right', 'rms_xyz_left', 'roll_right', 'roll_left', 'pitch_right', 'pitch_left']
   frame  right_leg_x  right_leg_y  right_leg_z  left_leg_x  left_leg_y  \
0      0     0.163942    -0.001378     0.045835    0.008039   -0.054908   
1      1    -0.190849     0.214363     0.051633   -0.541907    0.175742   
2      2    -0.054696    -0.127660    -0.076814    0.110754    0.134195   
3      3     0.184245    -0.046103     0.073129    0.013505   -0.010772   
4      4     0.302372    -0.010540     0.076275    0.210097    0.026873   

   left_leg_z  class  rms_xy_right  rms_xy_left  rms_yz_right  rms_yz_left  \
0    0.011806      0      0.115929     0.039240      0.032425     0.039713   
1    0.231347      0      0.202947     0.402833      0.155913    

In [3]:
import pandas as pd
import numpy as np
from scipy import integrate
from scipy.signal import find_peaks

# Load data from step 2
data = pd.read_csv('step2_data.csv')

# Define function to compute features for a data segment
def compute_features(data_segment, column_name):
    """Compute statistical features for a column in a data segment"""
    features = {}
    
    # Mean
    features[f'mean_{column_name}'] = data_segment[column_name].mean()
    
    # Standard deviation
    features[f'std_{column_name}'] = data_segment[column_name].std()
    
    # Min and Max
    features[f'min_{column_name}'] = data_segment[column_name].min()
    features[f'max_{column_name}'] = data_segment[column_name].max()
    
    # Area under the curve (AUC) - using trapezoidal rule
    features[f'auc_{column_name}'] = integrate.trapz(data_segment[column_name].values)
    
    # Number of peaks
    # Using prominence parameter to avoid detecting noise as peaks
    peaks, _ = find_peaks(data_segment[column_name].values, prominence=0.1)
    features[f'peaks_{column_name}'] = len(peaks)
    
    return features

# Identify data columns to compute features for (exclude frame and class)
feature_columns = [col for col in data.columns if col not in ['frame', 'class']]

# Initialize an empty list to store feature rows
feature_rows = []

# Process data in one-minute chunks (60 frames per minute)
for class_value in [0, 1]:  # Process boning and slicing separately
    class_data = data[data['class'] == class_value]
    
    # Group by minute (60 frames per minute)
    for i in range(0, len(class_data), 60):
        minute_data = class_data.iloc[i:i+60]
        if len(minute_data) < 60:  # Skip incomplete minutes
            continue
            
        # Initialize a feature dictionary for this minute
        minute_features = {'class': class_value}
        
        # Calculate features for each column
        for column in feature_columns:
            column_features = compute_features(minute_data, column)
            minute_features.update(column_features)
        
        feature_rows.append(minute_features)

# Create a feature DataFrame
feature_df = pd.DataFrame(feature_rows)

# Save feature dataset
feature_df.to_csv('step3_features.csv', index=False)

print(f"Feature dataset shape: {feature_df.shape}")
print(f"Number of features: {feature_df.shape[1] - 1}")  # Subtract 1 for class column
print(f"Number of samples: {feature_df.shape[0]}")
print(feature_df.head(2))

/var/folders/yr/mm0xskxn61jfrkxwp3y0mc7w0000gn/T/ipykernel_42156/4082646968.py:25: DeprecationWarning: 'scipy.integrate.trapz' is deprecated in favour of 'scipy.integrate.trapezoid' and will be removed in SciPy 1.14.0
  features[f'auc_{column_name}'] = integrate.trapz(data_segment[column_name].values)


Feature dataset shape: (1201, 109)
Number of features: 108
Number of samples: 1201
   class  mean_right_leg_x  std_right_leg_x  min_right_leg_x  max_right_leg_x  \
0      0         -0.001113         0.318366        -1.265566         0.695836   
1      0          0.091712         0.983078        -3.669763         2.055587   

   auc_right_leg_x  peaks_right_leg_x  mean_right_leg_y  std_right_leg_y  \
0         0.484026                 13          0.019689         0.193759   
1         6.062526                 15          0.099387         0.914605   

   min_right_leg_y  ...  min_pitch_right  max_pitch_right  auc_pitch_right  \
0        -0.490542  ...       -86.955827        83.471316       250.850399   
1        -1.934838  ...       -82.923057        80.049480       -40.772398   

   peaks_pitch_right  mean_pitch_left  std_pitch_left  min_pitch_left  \
0                 15         4.416288       48.345348      -86.152509   
1                 15        -1.625191       40.114711      -80.

In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.linear_model import SGDClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

# Load features
features = pd.read_csv('step3_features.csv')

# Split features and target
X = features.drop('class', axis=1)
y = features['class']

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Create a function to evaluate and print model performance
def evaluate_model(model, X_test, y_test, model_name):
    y_pred = model.predict(X_test)
    
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    print(f"\n{model_name} Results:")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")
    
    return {
        'model': model_name,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

# Initialize results lists for tables
svm_results = []
other_models_results = []

#
# SVM TRAINING APPROACHES
#

# 1. SVM with Train-Test Split (70/30)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.3, random_state=42)
print(f"Training set size: {X_train.shape[0]}, Test set size: {X_test.shape[0]}")

svm_basic = SVC(random_state=42)
svm_basic.fit(X_train, y_train)
result = evaluate_model(svm_basic, X_test, y_test, "SVM Basic (Train-Test Split)")
svm_results.append(result)

# 2. SVM with 10-fold Cross-Validation
svm_cv = SVC(random_state=42)
cv_scores = cross_val_score(svm_cv, X_scaled, y, cv=10)
print("\nSVM with 10-fold Cross-Validation:")
print(f"CV Accuracy: {cv_scores.mean():.4f} (±{cv_scores.std():.4f})")
svm_results.append({
    'model': 'SVM (10-fold CV)', 
    'accuracy': cv_scores.mean(), 
    'precision': None,  # Not directly available from cross_val_score
    'recall': None, 
    'f1': None
})

# 3. SVM with Hyperparameter Tuning
param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.01, 0.001],
    'kernel': ['rbf', 'linear']
}
grid_search = GridSearchCV(SVC(random_state=42), param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_train, y_train)

print("\nSVM Hyperparameter Tuning Results:")
print(f"Best parameters: {grid_search.best_params_}")
print(f"Best cross-validation score: {grid_search.best_score_:.4f}")

best_svm = grid_search.best_estimator_
result = evaluate_model(best_svm, X_test, y_test, "SVM Tuned (Train-Test Split)")
svm_results.append(result)

# 4. SVM with Feature Selection
selector = SelectKBest(f_classif, k=10)
X_train_selected = selector.fit_transform(X_train, y_train)
X_test_selected = selector.transform(X_test)

# Get selected feature names for reference
selected_indices = selector.get_support(indices=True)
selected_features = X.columns[selected_indices]
print("\nSelected Features:")
print(selected_features.tolist())

best_svm_fs = SVC(**grid_search.best_params_, random_state=42)
best_svm_fs.fit(X_train_selected, y_train)
result = evaluate_model(best_svm_fs, X_test_selected, y_test, "SVM Tuned with Feature Selection")
svm_results.append(result)

# 5. SVM with PCA
pca = PCA(n_components=10)
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

print(f"\nPCA Explained Variance: {sum(pca.explained_variance_ratio_):.4f}")

best_svm_pca = SVC(**grid_search.best_params_, random_state=42)
best_svm_pca.fit(X_train_pca, y_train)
result = evaluate_model(best_svm_pca, X_test_pca, y_test, "SVM Tuned with PCA")
svm_results.append(result)

#
# OTHER ML MODELS
#

# SGD Classifier
sgd = SGDClassifier(max_iter=1000, random_state=42)
sgd.fit(X_train, y_train)
result = evaluate_model(sgd, X_test, y_test, "SGD Classifier")
other_models_results.append(result)

# Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
result = evaluate_model(rf, X_test, y_test, "Random Forest")
other_models_results.append(result)

# MLP (Neural Network)
mlp = MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=1000, random_state=42)
mlp.fit(X_train, y_train)
result = evaluate_model(mlp, X_test, y_test, "MLP Neural Network")
other_models_results.append(result)

# Create summary tables
svm_results_df = pd.DataFrame(svm_results)
other_models_df = pd.DataFrame(other_models_results)

print("\nSVM Models Summary Table:")
print(svm_results_df)

print("\nOther Models Summary Table:")
print(other_models_df)

# Save results to CSV for inclusion in report
svm_results_df.to_csv('svm_results_table.csv', index=False)
other_models_df.to_csv('other_models_table.csv', index=False)

Training set size: 840, Test set size: 361

SVM Basic (Train-Test Split) Results:
Accuracy: 0.8975
Precision: 0.8571
Recall: 0.6585
F1 Score: 0.7448

SVM with 10-fold Cross-Validation:
CV Accuracy: 0.8951 (±0.0474)

SVM Hyperparameter Tuning Results:
Best parameters: {'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}
Best cross-validation score: 0.9000

SVM Tuned (Train-Test Split) Results:
Accuracy: 0.9058
Precision: 0.8636
Recall: 0.6951
F1 Score: 0.7703

Selected Features:
['peaks_right_leg_x', 'peaks_right_leg_z', 'peaks_left_leg_x', 'peaks_left_leg_y', 'peaks_left_leg_z', 'peaks_rms_xy_right', 'peaks_rms_zx_right', 'peaks_rms_zx_left', 'peaks_rms_xyz_right', 'peaks_pitch_right']

SVM Tuned with Feature Selection Results:
Accuracy: 0.8560
Precision: 0.7027
Recall: 0.6341
F1 Score: 0.6667

PCA Explained Variance: 0.7258

SVM Tuned with PCA Results:
Accuracy: 0.9003
Precision: 0.8382
Recall: 0.6951
F1 Score: 0.7600

SGD Classifier Results:
Accuracy: 0.8698
Precision: 0.7160
Recall: 0.7073


In [5]:
# Analyze results to select the best model
best_svm_model = svm_results_df.loc[svm_results_df['accuracy'].idxmax()]
best_overall_model = pd.concat([svm_results_df, other_models_df]).loc[pd.concat([svm_results_df, other_models_df])['accuracy'].idxmax()]

print("\nBest SVM Model:")
print(best_svm_model)

print("\nBest Overall Model:")
print(best_overall_model)

print("\nStep 5: Model Selection Analysis")
print("1. Best SVM model is", best_svm_model['model'], "with accuracy of", best_svm_model['accuracy'])
print("   This is due to [your analysis here - e.g., good balance of complexity and performance]")

print("2. Best overall model is", best_overall_model['model'], "with accuracy of", best_overall_model['accuracy'])
print("   This is due to [your analysis here - e.g., better ability to capture patterns]")


Best SVM Model:
model        SVM Tuned (Train-Test Split)
accuracy                         0.905817
precision                        0.863636
recall                           0.695122
f1                                0.77027
Name: 2, dtype: object

Best Overall Model:
                          model  accuracy  precision    recall        f1
2  SVM Tuned (Train-Test Split)  0.905817   0.863636  0.695122  0.770270
2            MLP Neural Network  0.880886   0.767123  0.682927  0.722581

Step 5: Model Selection Analysis
1. Best SVM model is SVM Tuned (Train-Test Split) with accuracy of 0.9058171745152355
   This is due to [your analysis here - e.g., good balance of complexity and performance]
2. Best overall model is 2    SVM Tuned (Train-Test Split)
2              MLP Neural Network
Name: model, dtype: object with accuracy of 2    0.905817
2    0.880886
Name: accuracy, dtype: float64
   This is due to [your analysis here - e.g., better ability to capture patterns]
